## Manual function tool calling Sample

### Setup Foundry Project Client

In [1]:
from typing import Any, Callable, Set, List
import re
import os
import time
import json
from azure.ai.projects import AIProjectClient
from azure.ai.agents.telemetry import trace_function
from azure.ai.agents.models import (
    FunctionTool,
    RequiredFunctionToolCall,
    SubmitToolOutputsAction,
    BingGroundingTool,
    ToolOutput,
    ThreadMessage,
    ToolSet,
    MessageRole,
)
import azure.ai.agents as agentslib
import azure.ai.projects as projectslib
from opentelemetry import trace
from azure.monitor.opentelemetry import configure_azure_monitor
from dotenv import load_dotenv, find_dotenv

# Your custom Python functions (for "fetch_datetime", etc.)
# from utils.enterprise_functions import enterprise_fns

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print("please login first")

Environment and authentication OK


In [2]:
# new AI Foundry Project resource endpoint / old azure ai services endpoint from the hub/project
project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
    # api_version=os.environ["PROJECT_API_VERSION"]
)
print("project_client api version:", project_client._config.api_version)
print(f"azure-ai-agents version: {agentslib.__version__}")
print(f"azure-ai-projects version: {projectslib.__version__}")

project_client api version: 2025-05-15-preview
azure-ai-agents version: 1.2.0b5
azure-ai-projects version: 1.1.0b4


### Set up the helper

* adding custom function 

In [3]:
def print_messages(messages: List[ThreadMessage]) -> None:
    """Helper function to print messages"""
    for message in messages:
        print(f"Role: {message.role}")
        print(f"Content: {message.content[0].text.value}")
        print("-" * 40)

In [4]:
def process_agent_run(thread_id: str, run_id: str, functions: FunctionTool) -> None:
    """Helper function to process an agent run and handle tool calls"""
    run = project_client.agents.runs.get(thread_id=thread_id, run_id=run_id)
    
    while run.status in ["queued", "in_progress", "requires_action"]:
        time.sleep(1)
        run = project_client.agents.runs.get(thread_id=thread_id, run_id=run_id)
        if run.status == "requires_action" and isinstance(run.required_action, SubmitToolOutputsAction):
            tool_calls = run.required_action.submit_tool_outputs.tool_calls
            if not tool_calls:
                print("No tool calls provided - cancelling run")
                project_client.agents.runs.cancel(thread_id=thread_id, run_id=run_id)
                break

            tool_outputs = []
            for tool_call in tool_calls:
                if isinstance(tool_call, RequiredFunctionToolCall):
                    try:
                        output = functions.execute(tool_call)
                        tool_outputs.append(
                            ToolOutput(tool_call_id=tool_call.id, output=output)
                        )
                    except Exception as e:
                        print(f"Error executing tool_call {tool_call.id}: {e}")

            if tool_outputs:
                project_client.agents.runs.submit_tool_outputs(
                    thread_id=thread_id,
                    run_id=run_id,
                    tool_outputs=tool_outputs
                )

        print(f"Current run status: {run.status}")
    
    return run

### Setup agent functions

In [5]:
try:
    bing_connection = project_client.connections.get(name=settings.bing_connection_name)
    conn_id = bing_connection.id
    bing_tool = BingGroundingTool(
        connection_id=conn_id,
        count=7, # number of search results to return in the response
        set_lang="en", # language to use for user interface strings
        market="en-us", # The market where the results come from.
        freshness="2025-11-02", # YYYY-MM-DD The date or date range used for filtering search results.
    )
    print("bing > connected")
except Exception:
    bing_tool = None
    print("bing failed > no connection found or permission issue")

bing > connected


In [6]:
instructions = (
    "You are a helpful enterprise assistant at Contoso. "
    "You have access to following tools. \n\n"
    "## Tools:\n"
    " * bing_grounding: get information about latest news from the public web.\n"
    "\n"
    "## Instructions:\n"
    "You can use the all the tools to answer questions\n"
    "\n"
    "## Guidelines:\n"
    "Provide well-structured and professional answers. Always return the grounding links for citations."
)

AGENT_NAME = "web-search-agent-manual"
found_agent = None
all_agents_list = project_client.agents.list_agents()
for a in all_agents_list:
    if a.name == AGENT_NAME:
        found_agent = a
        break

model_name = settings.model_deployment_name

In [ ]:
# Initialize functions
user_functions: Set[Callable[..., Any]] = {}
functions = FunctionTool(functions=user_functions)

class LoggingToolSet(ToolSet):
    def execute_tool_calls(self, tool_calls: List[Any]) -> List[dict]:
        """
        Execute the upstream calls, printing only two lines per function:
        1) The function name + its input arguments
        2) The function name + its output result
        """

        # For each function call, print the input arguments
        for c in tool_calls:
            if hasattr(c, "function") and c.function:
                fn_name = c.function.name
                fn_args = c.function.arguments
                print(f"{fn_name} inputs > {fn_args} (id:{c.id})")

        # Execute the tool calls (superclass logic)
        raw_outputs = super().execute_tool_calls(tool_calls)

        # Print the output of each function call
        for item in raw_outputs:
            print(f"output > {item['output']}")

        return raw_outputs

toolset = LoggingToolSet()
toolset.add(user_functions)
toolset.add(bing_tool)

In [ ]:
if found_agent:
    # print(found_agent)
    # Update the existing agent to use new tools
    agent = project_client.agents.update_agent(
        agent_id=found_agent.id,
        model=model_name,
        instructions=instructions,
        toolset=bing_tool.definitions
    )
    print(f"reusing agent > {agent.name} (id: {agent.id})")
else:
    agent = project_client.agents.create_agent(
        model=model_name,
        name=AGENT_NAME,
        instructions=instructions,
        toolset=bing_tool.definitions
    )
    print(f"creating agent > {agent.name} (id: {agent.id})")

### Sequential Agent Runs inside Tracing Scope

* Create two agents: weather assistant agent, and temperature conversion agent (Celsius and Fahrenheit)
* first run the weather assistant agent
* check output contains celsius, then run the temperature conversion agent subsequently
* use the same thread for both agent, so that the agent see the whole history (not recommended)
* return the messages from both agents run

In [ ]:
with tracer.start_as_current_span(scenario):
    with project_client:
        # Create two agents with different roles
        # Create the weather assistant agent
        weather_agent = project_client.agents.create_agent(
            model=settings.model_deployment_name,
            name="weather-assistant",
#             instructions="""You are a helpful weather assistant. Follow these steps:
# 1. When asked about weather, politely acknowledge the request
# 2. Use the fetch_weather function to get the weather information
# 3. Present the weather information in a friendly, conversational way
# 4. If the temperature is in Celsius, mention that you'll ask for a conversion to Fahrenheit
# Be descriptive and natural in your responses.""",
            instructions="""You are a helpful weather assistant. Follow these steps:
1. When asked about weather, politely acknowledge the request
2. Use the fetch_weather function to get the weather information
3. Present the weather information in a friendly, conversational way
4. If the temperature is in Celsius, report back only the Celsius temperature
Be descriptive and natural in your responses.""",
            tools=functions.definitions,
        )
        print(f"Created weather agent, ID: {weather_agent.id}")

        # Create the temperature conversion agent
        conversion_agent = project_client.agents.create_agent(
            model=settings.model_deployment_name,
            name="conversion-assistant",
            instructions="""You are a helpful temperature conversion assistant. Follow these steps:
1. When asked to convert a temperature, acknowledge the request
2. Extract the temperature value from the previous messages
3. Use the convert_temperature function to perform the conversion
4. Present both temperatures in a clear, friendly way
Be detailed and explain the conversion clearly.""",
            tools=functions.definitions,
        )
        print(f"Created conversion agent, ID: {conversion_agent.id}")

        # Try block content for conversation display
        try:
            # Create a thread for the conversation
            thread = project_client.agents.threads.create()
            print(f"Created thread, ID: {thread.id}")

            # User asks about weather with a clear request
            message = project_client.agents.messages.create(
                thread_id=thread.id,
                role="user",
                # content="Can you tell me what the weather is like in New York today? I'd like to know the temperature in both Celsius and Fahrenheit, please.",
                content="Can you tell me what the weather is like in New York today? I'd like to know the temperature in Celsius °C please.",
            )
            print(f"Created initial user message, ID: {message.id}")

            # Start with the weather agent for the initial response
            weather_run = project_client.agents.runs.create(
                thread_id=thread.id,
                agent_id=weather_agent.id
            )
            print(f"Started weather agent run, ID: {weather_run.id}")

            # Process the weather agent's run
            weather_run = process_agent_run(thread.id, weather_run.id, functions)
            print(f"Weather agent run completed with status: {weather_run.status}")

            # Check messages for Celsius temperatures
            messages = list(project_client.agents.messages.list(thread_id=thread.id))
            print(f"Total messages in thread: {len(messages)}")
            # print_messages(messages)

            if check_for_celsius_in_messages(messages):
                
                # Create a message for the conversion agent with context
                conversion_msg = project_client.agents.messages.create(
                    thread_id=thread.id,
                    role="user",
                    content="Could you help convert this Celsius temperature to Fahrenheit so we can see both? Please explain the conversion."
                )
                print(f"Created conversion request message, ID: {conversion_msg.id}")

                # Start the conversion agent's run
                conversion_run = project_client.agents.runs.create(
                    thread_id=thread.id,
                    agent_id=conversion_agent.id
                )
                print(f"Started conversion agent run, ID: {conversion_run.id}")

                # Process the conversion agent's run
                conversion_run = process_agent_run(thread.id, conversion_run.id, functions)
                print(f"Conversion agent run completed with status: {conversion_run.status}")

            # Display the full conversation
            print("\nFull conversation:")
            final_messages = list(project_client.agents.messages.list(thread_id=thread.id))
            # print_messages(final_messages)

            final_messages.reverse()  # Show messages in chronological order
            for msg in final_messages:
                try:
                    if hasattr(msg, 'content'):
                        role = msg.role if hasattr(msg, 'role') else 'system'
                        print(f"{role}: {msg.content}")
                except Exception as e:
                    print(f"Error printing message: {e}")

        finally:
            # Clean up resources
            print("\nCleaning up resources...")
            project_client.agents.delete_agent(weather_agent.id)
            project_client.agents.delete_agent(conversion_agent.id)
            print("Agents deleted successfully")